# Benchmark · Síntesis de voz (TTS) · Bloque 1

**Responsable:** …  
**Fecha:** …

> Comparativa de modelos de síntesis de voz para el briefing de audio de Hawk & Dove.

In [ ]:
# Configuración común: rutas del proyecto y config.yaml
from src.config import ROOT, EVENTS_DIR, HISTORY_DIR, RAW_DIR, load_config, event_dir

cfg = load_config()
ref_date = cfg["reference_event"]
ref_dir = event_dir(ref_date)
print(f"Evento de referencia: {ref_date}")

## 1. Objetivo

Elegir el modelo de TTS que mejor sintetiza el briefing de audio del informe Hawk & Dove.

Criterios: naturalidad de la voz, pronunciación correcta de términos financieros (ECB, basis points, inflation), y velocidad de generación.

## 2. Candidatos

| Modelo | Tamaño | Licencia | Por qué se incluye |
|---|---|---|---|
| hexgrad/Kokoro-82M | 82M | MIT | Ligero, calidad profesional, muy rápido |
| parler-tts/parler-tts-mini-v1 | 880M | Apache 2.0 | Controlable por prompt de texto, mayor expresividad |

## 3. Datos de referencia

Texto de prueba: el `briefing_tts` de `summary.json` del evento de referencia, que contiene terminología financiera del BCE.

La evaluación es subjetiva (escucha humana) + métricas automáticas de calidad.

In [ ]:
# Cargar texto de prueba
import json

summary_path = ref_dir / "summary.json"
with open(summary_path, encoding="utf-8") as f:
    summary = json.load(f)

texto_tts = summary.get("briefing_tts", "")
if not texto_tts:
    # Texto de ejemplo si summary.json aún no tiene briefing_tts
    texto_tts = (
        "The European Central Bank's Governing Council decided to keep the three key "
        "interest rates unchanged at its September 2026 meeting. President Lagarde "
        "noted that inflation has continued to decline toward the two percent target, "
        "but domestic price pressures remain elevated due to strong wage growth. "
        "The overall tone was slightly hawkish, ranking in the 72nd percentile "
        "compared to previous conferences under Lagarde's presidency."
    )
print(f"Texto: {len(texto_tts)} caracteres")
print(texto_tts[:200], "...")


## 4. Métrica

- **MOS subjetivo:** escucha de 3 evaluadores, puntuación 1-5 (naturalidad, claridad, pronunciación).
- **RTF (Real-Time Factor):** duración_audio / tiempo_generación. RTF < 1 = más rápido que tiempo real.
- **VRAM:** consumo de memoria GPU.

In [ ]:
# Generar audio con Kokoro
import time
import soundfile as sf
from src.audio.models.tts_backends import crear_backend_tts, liberar_memoria

backend = crear_backend_tts("kokoro")

t0 = time.time()
audio, sr = backend.sintetizar(texto_tts)
t_gen = time.time() - t0

duracion_audio = len(audio) / sr
rtf = duracion_audio / t_gen

print(f"Duración audio: {duracion_audio:.1f}s")
print(f"Tiempo generación: {t_gen:.1f}s")
print(f"RTF: {rtf:.1f}x tiempo real")
print(f"VRAM: {backend.memoria_gb()} GB")

# Guardar para escucha
sf.write(ref_dir / "briefing_kokoro.wav", audio, sr)
liberar_memoria()


## 5. Resultados

Comparar RTF, VRAM y calidad subjetiva de cada candidato.

In [ ]:
# Comparativa (cuando Parler-TTS esté implementado)
candidatos = [
    "kokoro",
    # "parler-tts/parler-tts-mini-v1",  # activar cuando se implemente
]

resultados = []
for modelo_id in candidatos:
    backend = crear_backend_tts(modelo_id)
    t0 = time.time()
    audio, sr = backend.sintetizar(texto_tts)
    t_gen = time.time() - t0
    duracion = len(audio) / sr
    resultados.append({
        "modelo": modelo_id,
        "duracion_s": round(duracion, 1),
        "tiempo_gen_s": round(t_gen, 1),
        "rtf": round(duracion / t_gen, 1),
        "vram_gb": backend.memoria_gb(),
    })
    liberar_memoria()

import pandas as pd
df = pd.DataFrame(resultados)
print(df.to_markdown(index=False))


## 6. Latencia y memoria

Hardware utilizado.

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


## 7. Decisión

Modelo elegido y argumento. Anotar el ganador en `config.yaml` bajo `models.tts`.

### Escucha del briefing generado

Escuchar los archivos `.wav` generados en `data/events/<fecha>/` para comparar la calidad subjetiva.